# Day 4 — Feature Engineering
## Career Progression and Promotion Gap Analysis

Create meaningful features from the cleaned HR dataset for promotion gaps, role stagnation, manager stability, career stage, and training.

> Features are analytical signals, not conclusions.

## Workflow
1. Load Cleaned Dataset
2. Feature Engineering Plan
3. Promotion Gap Ratio
4. Role Stagnation Ratio
5. Manager Stability Ratio
6. Career Stage
7. Promotion Gap Group
8. Role Duration Group
9. Training Group
10. Review New Features
11. Validate Features
12. Feature Summary
13. Final Dataset Check
14. Automatically Save Dataset, Tables & Report

# 1. Load Libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)


# 2. Load Cleaned Dataset

In [2]:
candidate_paths = [
    Path("../data/cleaned/Palo Alto Networks_cleaned.csv"),
    Path("../data/cleaned_palo_alto.csv"),
    Path("../data/Palo Alto Networks_cleaned.csv"),
    Path("Palo Alto Networks_cleaned.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Cleaned dataset not found. Run Day 2 first or update candidate_paths."
    )

df = pd.read_csv(data_path)

print("Dataset:", data_path)
print("Dataset shape:", df.shape)
display(df.head())


Dataset: ../data/cleaned/Palo Alto Networks_cleaned.csv
Dataset shape: (1470, 31)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Yes,11,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,No,23,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Yes,15,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Yes,11,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,No,12,3,4,1,6,3,3,2,2,2,2


# 3. Feature Engineering Plan

| Feature | Purpose |
|---|---|
| `PromotionGapRatio` | Promotion gap relative to company tenure |
| `RoleStagnationRatio` | Time in current role relative to company tenure |
| `ManagerStabilityRatio` | Current-manager tenure relative to company tenure |
| `CareerStage` | Employee career-stage group |
| `PromotionGapGroup` | Promotion-gap category |
| `RoleDurationGroup` | Current-role duration category |
| `TrainingGroup` | Training activity category |

For `YearsAtCompany = 0`, ratio calculations use safe handling.

# 4. Promotion Gap Ratio

**Formula:** `YearsSinceLastPromotion / YearsAtCompany`

Example: 7 years since promotion / 10 years at company = 0.70.

A higher ratio means a larger portion of company tenure has passed since the last promotion.

In [3]:
df["PromotionGapRatio"] = np.divide(
    df["YearsSinceLastPromotion"],
    df["YearsAtCompany"],
    out=np.zeros(len(df), dtype=float),
    where=df["YearsAtCompany"].to_numpy() > 0
)

display(df[
    ["YearsAtCompany", "YearsSinceLastPromotion", "PromotionGapRatio"]
].head(10))


,YearsAtCompany,YearsSinceLastPromotion,PromotionGapRatio
0,6,0,0.000000
1,10,1,0.100000
2,0,0,0.000000
3,8,3,0.375000
4,2,2,1.000000
5,7,3,0.428571
6,1,0,0.000000
7,1,0,0.000000
8,9,1,0.111111
9,7,7,1.000000


# 5. Role Stagnation Ratio

**Formula:** `YearsInCurrentRole / YearsAtCompany`

A higher value means the employee has spent a larger proportion of company tenure in the current role.

A high ratio does not automatically mean negative career stagnation.

In [4]:
df["RoleStagnationRatio"] = np.divide(
    df["YearsInCurrentRole"],
    df["YearsAtCompany"],
    out=np.zeros(len(df), dtype=float),
    where=df["YearsAtCompany"].to_numpy() > 0
)

display(df[
    ["YearsAtCompany", "YearsInCurrentRole", "RoleStagnationRatio"]
].head(10))


,YearsAtCompany,YearsInCurrentRole,RoleStagnationRatio
0,6,4,0.666667
1,10,7,0.700000
2,0,0,0.000000
3,8,7,0.875000
4,2,2,1.000000
5,7,7,1.000000
6,1,0,0.000000
7,1,0,0.000000
8,9,7,0.777778
9,7,7,1.000000


# 6. Manager Stability Ratio

**Formula:** `YearsWithCurrManager / YearsAtCompany`

A higher value means the employee has spent a larger proportion of company tenure with the current manager.

In [5]:
df["ManagerStabilityRatio"] = np.divide(
    df["YearsWithCurrManager"],
    df["YearsAtCompany"],
    out=np.zeros(len(df), dtype=float),
    where=df["YearsAtCompany"].to_numpy() > 0
)

display(df[
    ["YearsAtCompany", "YearsWithCurrManager", "ManagerStabilityRatio"]
].head(10))


,YearsAtCompany,YearsWithCurrManager,ManagerStabilityRatio
0,6,5,0.833333
1,10,7,0.700000
2,0,0,0.000000
3,8,0,0.000000
4,2,2,1.000000
5,7,6,0.857143
6,1,0,0.000000
7,1,0,0.000000
8,9,8,0.888889
9,7,7,1.000000


# 7. Career Stage

Initial analytical groups:

- **Early Career** → 0–2 years
- **Developing** → 3–5 years
- **Established** → 6–10 years
- **Experienced** → 11+ years

These are descriptive categories, not official HR policies.

In [6]:
df["CareerStage"] = pd.cut(
    df["YearsAtCompany"],
    bins=[-1, 2, 5, 10, np.inf],
    labels=["Early Career", "Developing", "Established", "Experienced"]
)

display(df[["YearsAtCompany", "CareerStage"]].head(10))


,YearsAtCompany,CareerStage
0,6,Established
1,10,Established
2,0,Early Career
3,8,Established
4,2,Early Career
5,7,Established
6,1,Early Career
7,1,Early Career
8,9,Established
9,7,Established


# 8. Promotion Gap Group

Initial categories:

- **Recent Promotion** → 0–1 years
- **Moderate Gap** → 2–4 years
- **Long Promotion Gap** → 5+ years

In [7]:
df["PromotionGapGroup"] = pd.cut(
    df["YearsSinceLastPromotion"],
    bins=[-1, 1, 4, np.inf],
    labels=["Recent Promotion", "Moderate Gap", "Long Promotion Gap"]
)

display(df[
    ["YearsSinceLastPromotion", "PromotionGapGroup"]
].head(10))


,YearsSinceLastPromotion,PromotionGapGroup
0,0,Recent Promotion
1,1,Recent Promotion
2,0,Recent Promotion
3,3,Moderate Gap
4,2,Moderate Gap
5,3,Moderate Gap
6,0,Recent Promotion
7,0,Recent Promotion
8,1,Recent Promotion
9,7,Long Promotion Gap


# 9. Role Duration Group

Initial categories:

- **New in Role** → 0–2 years
- **Established** → 3–5 years
- **Long Time in Role** → 6+ years

In [8]:
df["RoleDurationGroup"] = pd.cut(
    df["YearsInCurrentRole"],
    bins=[-1, 2, 5, np.inf],
    labels=["New in Role", "Established", "Long Time in Role"]
)

display(df[
    ["YearsInCurrentRole", "RoleDurationGroup"]
].head(10))


,YearsInCurrentRole,RoleDurationGroup
0,4,Established
1,7,Long Time in Role
2,0,New in Role
3,7,Long Time in Role
4,2,New in Role
5,7,Long Time in Role
6,0,New in Role
7,0,New in Role
8,7,Long Time in Role
9,7,Long Time in Role


# 10. Training Group

Initial categories:

- **No Training** → 0
- **Low Training** → 1–2
- **Moderate Training** → 3–4
- **High Training** → 5+

In [9]:
df["TrainingGroup"] = pd.cut(
    df["TrainingTimesLastYear"],
    bins=[-1, 0, 2, 4, np.inf],
    labels=["No Training", "Low Training", "Moderate Training", "High Training"]
)

display(df[
    ["TrainingTimesLastYear", "TrainingGroup"]
].head(10))


,TrainingTimesLastYear,TrainingGroup
0,0,No Training
1,3,Moderate Training
2,3,Moderate Training
3,3,Moderate Training
4,3,Moderate Training
5,2,Low Training
6,3,Moderate Training
7,2,Low Training
8,2,Low Training
9,3,Moderate Training


# 11. Review All New Features

In [10]:
new_features = [
    "PromotionGapRatio",
    "RoleStagnationRatio",
    "ManagerStabilityRatio",
    "CareerStage",
    "PromotionGapGroup",
    "RoleDurationGroup",
    "TrainingGroup"
]

display(df[new_features].head(10))


,PromotionGapRatio,RoleStagnationRatio,ManagerStabilityRatio,CareerStage,PromotionGapGroup,RoleDurationGroup,TrainingGroup
0,0.000000,0.666667,0.833333,Established,Recent Promotion,Established,No Training
1,0.100000,0.700000,0.700000,Established,Recent Promotion,Long Time in Role,Moderate Training
2,0.000000,0.000000,0.000000,Early Career,Recent Promotion,New in Role,Moderate Training
3,0.375000,0.875000,0.000000,Established,Moderate Gap,Long Time in Role,Moderate Training
4,1.000000,1.000000,1.000000,Early Career,Moderate Gap,New in Role,Moderate Training
5,0.428571,1.000000,0.857143,Established,Moderate Gap,Long Time in Role,Low Training
6,0.000000,0.000000,0.000000,Early Career,Recent Promotion,New in Role,Moderate Training
7,0.000000,0.000000,0.000000,Early Career,Recent Promotion,New in Role,Low Training
8,0.111111,0.777778,0.888889,Established,Recent Promotion,Long Time in Role,Low Training
9,1.000000,1.000000,1.000000,Established,Long Promotion Gap,Long Time in Role,Moderate Training


# 12. Validate Ratio Features

Check missing, infinite, negative, and above-one values.

**Important:** Because Day 2 documented logical career-duration inconsistencies, values above 1 are reported for investigation rather than automatically changed.

In [11]:
ratio_features = [
    "PromotionGapRatio",
    "RoleStagnationRatio",
    "ManagerStabilityRatio"
]

ratio_validation = []

for col in ratio_features:
    ratio_validation.append({
        "Feature": col,
        "Missing_Count": int(df[col].isna().sum()),
        "Infinite_Count": int(np.isinf(df[col]).sum()),
        "Below_Zero_Count": int((df[col] < 0).sum()),
        "Above_One_Count": int((df[col] > 1).sum()),
        "Minimum": round(df[col].min(), 4),
        "Maximum": round(df[col].max(), 4)
    })

ratio_validation = pd.DataFrame(ratio_validation)
display(ratio_validation)


,Feature,Missing_Count,Infinite_Count,Below_Zero_Count,Above_One_Count,Minimum,Maximum
0,PromotionGapRatio,0,0,0,0,0.0,1.0
1,RoleStagnationRatio,0,0,0,0,0.0,1.0
2,ManagerStabilityRatio,0,0,0,0,0.0,1.0


# 13. Statistical Summary of New Numerical Features

In [12]:
ratio_summary = df[ratio_features].describe().T.round(4)
display(ratio_summary)


,count,mean,std,min,25%,50%,75%,max
PromotionGapRatio,1470.0,0.2902,0.3405,0.0,0.0000,0.1667,0.5000,1.0
RoleStagnationRatio,1470.0,0.5782,0.3320,0.0,0.3525,0.6667,0.8333,1.0
ManagerStabilityRatio,1470.0,0.5586,0.3347,0.0,0.3333,0.6667,0.8000,1.0


# 14. Distribution of New Categorical Features

In [13]:
group_features = [
    "CareerStage",
    "PromotionGapGroup",
    "RoleDurationGroup",
    "TrainingGroup"
]

group_distribution = []

for col in group_features:
    counts = df[col].value_counts(dropna=False)
    for category, count in counts.items():
        group_distribution.append({
            "Feature": col,
            "Category": str(category),
            "Employee_Count": int(count),
            "Percentage": round(count / len(df) * 100, 2)
        })

group_distribution_summary = pd.DataFrame(group_distribution)
display(group_distribution_summary)


,Feature,Category,Employee_Count,Percentage
0,CareerStage,Established,448,30.48
1,CareerStage,Developing,434,29.52
2,CareerStage,Early Career,342,23.27
3,CareerStage,Experienced,246,16.73
4,PromotionGapGroup,Recent Promotion,938,63.81
5,PromotionGapGroup,Moderate Gap,272,18.50
6,PromotionGapGroup,Long Promotion Gap,260,17.69
7,RoleDurationGroup,New in Role,673,45.78
8,RoleDurationGroup,Long Time in Role,522,35.51
9,RoleDurationGroup,Established,275,18.71


# 15. Check for Missing Values Created by Feature Engineering

In [14]:
feature_missing = (
    df[new_features]
    .isna()
    .sum()
    .reset_index()
)
feature_missing.columns = ["Feature", "Missing_Count"]

display(feature_missing)
print("Total missing in engineered features:",
      int(feature_missing["Missing_Count"].sum()))


,Feature,Missing_Count
0,PromotionGapRatio,0
1,RoleStagnationRatio,0
2,ManagerStabilityRatio,0
3,CareerStage,0
4,PromotionGapGroup,0
5,RoleDurationGroup,0
6,TrainingGroup,0


Total missing in engineered features: 0


# 16. Compare Original and Engineered Features

Original variables are retained. Later clustering will decide which variables are most useful and avoid unnecessary redundancy.

In [15]:
comparison_cols = [
    "YearsAtCompany",
    "YearsSinceLastPromotion",
    "PromotionGapRatio",
    "YearsInCurrentRole",
    "RoleStagnationRatio",
    "YearsWithCurrManager",
    "ManagerStabilityRatio"
]

display(df[comparison_cols].head(10))


,YearsAtCompany,YearsSinceLastPromotion,PromotionGapRatio,YearsInCurrentRole,RoleStagnationRatio,YearsWithCurrManager,ManagerStabilityRatio
0,6,0,0.000000,4,0.666667,5,0.833333
1,10,1,0.100000,7,0.700000,7,0.700000
2,0,0,0.000000,0,0.000000,0,0.000000
3,8,3,0.375000,7,0.875000,0,0.000000
4,2,2,1.000000,2,1.000000,2,1.000000
5,7,3,0.428571,7,1.000000,6,0.857143
6,1,0,0.000000,0,0.000000,0,0.000000
7,1,0,0.000000,0,0.000000,0,0.000000
8,9,1,0.111111,7,0.777778,8,0.888889
9,7,7,1.000000,7,1.000000,7,1.000000


# 17. Feature Summary

| Engineered Feature | Meaning |
|---|---|
| `PromotionGapRatio` | Relative size of the promotion gap |
| `RoleStagnationRatio` | Proportion of company tenure spent in current role |
| `ManagerStabilityRatio` | Proportion of company tenure with current manager |
| `CareerStage` | Career-stage category |
| `PromotionGapGroup` | Promotion-gap category |
| `RoleDurationGroup` | Current-role duration category |
| `TrainingGroup` | Training activity category |

These features are signals, not conclusions.

In [16]:
feature_summary = pd.DataFrame({
    "Feature": new_features,
    "Type": [
        "Numerical Ratio", "Numerical Ratio", "Numerical Ratio",
        "Categorical Group", "Categorical Group",
        "Categorical Group", "Categorical Group"
    ],
    "Meaning": [
        "Promotion gap relative to company tenure",
        "Current-role tenure relative to company tenure",
        "Current-manager tenure relative to company tenure",
        "Employee career-stage group",
        "Promotion-gap category",
        "Current-role duration category",
        "Training activity category"
    ]
})

display(feature_summary)


,Feature,Type,Meaning
0,PromotionGapRatio,Numerical Ratio,Promotion gap relative to company tenure
1,RoleStagnationRatio,Numerical Ratio,Current-role tenure relative to company tenure
2,ManagerStabilityRatio,Numerical Ratio,Current-manager tenure relative to company tenure
3,CareerStage,Categorical Group,Employee career-stage group
4,PromotionGapGroup,Categorical Group,Promotion-gap category
5,RoleDurationGroup,Categorical Group,Current-role duration category
6,TrainingGroup,Categorical Group,Training activity category


# 18. Final Dataset Check

In [17]:
print("Current dataset shape:", df.shape)
print("Number of new features:", len(new_features))

print("\nNew features:")
for feature in new_features:
    print("-", feature)

assert set(new_features).issubset(df.columns)
assert df[new_features].isna().sum().sum() == 0
assert np.isfinite(df[ratio_features].to_numpy()).all()

print("\nFeature engineering validation passed.")


Current dataset shape: (1470, 38)
Number of new features: 7

New features:
- PromotionGapRatio
- RoleStagnationRatio
- ManagerStabilityRatio
- CareerStage
- PromotionGapGroup
- RoleDurationGroup
- TrainingGroup

Feature engineering validation passed.


# 19. Automatically Save Dataset, Tables & Report

In [18]:
output_dir = Path("../outputs/day_4")
tables_dir = output_dir / "tables"
reports_dir = output_dir / "reports"
feature_dir = Path("../data/feature_engineered")

for folder in [tables_dir, reports_dir, feature_dir]:
    folder.mkdir(parents=True, exist_ok=True)

# 1. Feature-engineered dataset
output_path = feature_dir / "Palo Alto Networks_feature_engineered.csv"
df.to_csv(output_path, index=False)

# 2. Analysis tables
tables = {
    "ratio_validation": ratio_validation,
    "ratio_summary": ratio_summary,
    "group_distribution_summary": group_distribution_summary,
    "feature_missing": feature_missing,
    "feature_summary": feature_summary,
    "feature_preview": df[new_features].head(20),
    "feature_comparison": df[comparison_cols].head(20)
}

for name, table in tables.items():
    table.to_csv(tables_dir / f"{name}.csv", index=False)

# 3. Report
logical_note = (
    "Ratio values above 1 are reported for investigation because Day 2 "
    "documented logical career-duration inconsistencies."
)

report = f'''# Day 4 — Feature Engineering Report

## Input
{data_path}

## Dataset Shape
{df.shape}

## Engineered Features
{chr(10).join("- " + x for x in new_features)}

## Validation
- Missing values in engineered features: {int(df[new_features].isna().sum().sum())}
- Infinite ratio values: {int(np.isinf(df[ratio_features].to_numpy()).sum())}
- Negative ratio values: {int((df[ratio_features] < 0).sum().sum())}

## Important Note
{logical_note}

## Output
Feature-engineered dataset:
{output_path}

Tables:
{tables_dir}
'''

(reports_dir / "day_4_feature_engineering_report.md").write_text(
    report,
    encoding="utf-8"
)

print("Feature-engineered dataset saved:", output_path)
print("Tables saved:", tables_dir)
print("Report saved:", reports_dir / "day_4_feature_engineering_report.md")


Feature-engineered dataset saved: ../data/feature_engineered/Palo Alto Networks_feature_engineered.csv
Tables saved: ../outputs/day_4/tables
Report saved: ../outputs/day_4/reports/day_4_feature_engineering_report.md
